In [ ]:
!{sys.executable} -m pip install --user openpyxl pandas numpy lxml unidecode
# # Install Selenium
!pip install selenium
# # Install Webdriver Manager
!pip install webdriver-manager
# Install Google Chrome Stable for headless execution in Colab
!apt-get update
# Install required packages for adding repositories
!apt-get install -y software-properties-common wget gnupg
# Download and add Google Chrome's signing key
!wget -q -O - https://dl.google.com/linux/linux_signing_key.pub | apt-key add -
# Add Google Chrome's repository to the APT sources
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list
# Update package lists after adding the new repository
!apt-get update
# Install google-chrome-stable
!apt-get install -y google-chrome-stable

/bin/bash: line 1: {sys.executable}: command not found
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 9.5 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy 

In [ ]:
!pip install requests beautifulsoup4

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# INPE-01 (seccion reinsercion)

In [ ]:
import pandas as pd
import time
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# 1. CONFIGURACIÓN
opciones = Options()
opciones.add_argument("--headless=new")
opciones.add_argument("--no-sandbox")
opciones.add_argument("--disable-dev-shm-usage")
opciones.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opciones)

url = "https://www.gob.pe/institucion/inpe/buscador?contenido=servicios&sheet=1&sort_by=none&tema=439-resocializacion"

try:
    print("Conectando al INPE (Simulando usuario real)...")
    driver.get(url)

    print("Esperando renderizado de la plataforma...")
    time.sleep(15)

    print("Extrayendo datos de la memoria del navegador...")
    script = "return JSON.stringify(window.initialData);"
    data_json_str = driver.execute_script(script)

    if data_json_str:
        data = json.loads(data_json_str)
        results = data['data']['attributes']['results']

        lista_final = []
        for item in results:
            lista_final.append({
                "ID": item.get('id'),
                "Título": item.get('name_with_parent'),
                "Resumen": item.get('content'),
                "Tipo": item.get('page_kind'),
                "Visitas": item.get('last_week_visits')
            })

        df = pd.DataFrame(lista_final)

        #

        # Configuramos Pandas para que no recorte el texto al imprimir
        pd.set_option('display.max_colwidth', None)

        print(f"\n¡CONSEGUIDO! Se extrajeron {len(df)} registros.")
        print("-" * 50)

        # Bucle para imprimir cada registro
        for index, row in df.iterrows():
            print(f"REGISTRO {index + 1}:")
            print(f"TÍTULO: {row['Título']}")
            print(f"TIPO: {row['Tipo']}")
            print(f"RESUMEN: {row['Resumen']}")
            print(f"VISITAS SEMANALES: {row['Visitas']}")
            print("-" * 50) # Separador visual entre noticias

        # ----------------------------------

        # Guardar evidencia
        df.to_excel("DATA_TESIS_FINAL.xlsx", index=False)
        print("\nArchivo 'DATA_TESIS_FINAL.xlsx' generado con éxito.")
    else:
        print("Error: La memoria del navegador no tiene la variable 'initialData'.")

except Exception as e:
    print(f"Error crítico: {e}")

finally:
    driver.quit()

Conectando al INPE (Simulando usuario real)...
Esperando renderizado de la plataforma...
Extrayendo datos de la memoria del navegador...

¡CONSEGUIDO! Se extrajeron 3 registros.
--------------------------------------------------
REGISTRO 1:
TÍTULO: Inscribir a una institución como unidad beneficiaria para recibir sentenciados a prestación de servicios comunitarios
TIPO: service
RESUMEN: Si perteneces a una institución pública o privada sin fines de lucro, puedes inscribir a tu organización como unidad beneficiaria para recibir sentenciados a prestar servicios comunitarios.  Cabe recalcar que tu inst ...
VISITAS SEMANALES: 5
--------------------------------------------------
REGISTRO 2:
TÍTULO: Solicitar autorización de asistencia religiosa a los establecimientos penitenciarios
TIPO: procedure
RESUMEN: Si eres miembro de un grupo o entidad religiosa y deseas realizar acompañamiento religioso a los internos en penales, cárceles o establecimientos penitenciarios, debes solicitar una autor

# INPE 02 (seccion noticias)

In [ ]:
import pandas as pd
import numpy as np
import time
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def configurar_driver():
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--window-size=1920,1080')
    # Añadimos estas dos para evitar el Connection Refused
    chrome_options.add_argument('--disable-browser-side-navigation')
    chrome_options.add_argument('--disable-gpu')

    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=chrome_options)

# URL base
base_url = "https://www.gob.pe/institucion/inpe/buscador?contenido=noticias&institucion=inpe&sort_by=recent&sheet="

all_noticias = []
path_folder = './Modulo_Tesis_Scrapping'
os.makedirs(path_folder, exist_ok=True)

# Bucle dinámico
for page in np.arange(1, 41):
    print(f"Iniciando extracción de Página {page}...")
    driver = configurar_driver()
    wait = WebDriverWait(driver, 30)

    try:
        # Construimos la URL de la página directamente
        driver.get(f"{base_url}{page}")

        # Espera activa
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "article")))
        time.sleep(5) # Pausa de cortesía al servidor

        bloques = driver.find_elements(By.TAG_NAME, 'article')
        print(f"Página {page}: Encontrados {len(bloques)} artículos.")

        for bloque in bloques:
            try:
                tag_a = bloque.find_element(By.CLASS_NAME, "link-principal")
                resumen = bloque.find_element(By.XPATH, ".//div[last()]").text.strip()

                all_noticias.append({
                    "PAGINA": page,
                    "TITULO": tag_a.text.strip(),
                    "ENLACE": tag_a.get_attribute('href'),
                    "RESUMEN": resumen
                })
            except:
                continue

    except Exception as e:
        print(f"Error en página {page}: {e}")
    finally:
        # IMPORTANTE: Cerramos el driver en cada ciclo para liberar memoria y puerto
        driver.quit()
        time.sleep(2)

# Guardado final
if all_noticias:
    df_total = pd.DataFrame(all_noticias)
    df_total.to_excel(os.path.join(path_folder, 'Data_INPE_Completa.xlsx'), index=False)
    print(f"\n¡PROCESO FINALIZADO! Total noticias: {len(all_noticias)}")
    print(df_total.head())

Iniciando extracción de Página 1...
Página 1: Encontrados 25 artículos.
Iniciando extracción de Página 2...
Página 2: Encontrados 25 artículos.
Iniciando extracción de Página 3...
Página 3: Encontrados 25 artículos.
Iniciando extracción de Página 4...
Página 4: Encontrados 25 artículos.
Iniciando extracción de Página 5...
Página 5: Encontrados 25 artículos.
Iniciando extracción de Página 6...
Página 6: Encontrados 25 artículos.
Iniciando extracción de Página 7...
Página 7: Encontrados 25 artículos.
Iniciando extracción de Página 8...
Página 8: Encontrados 25 artículos.
Iniciando extracción de Página 9...
Página 9: Encontrados 25 artículos.
Iniciando extracción de Página 10...
Página 10: Encontrados 25 artículos.
Iniciando extracción de Página 11...
Página 11: Encontrados 25 artículos.
Iniciando extracción de Página 12...
Página 12: Encontrados 25 artículos.
Iniciando extracción de Página 13...
Página 13: Encontrados 25 artículos.
Iniciando extracción de Página 14...
Página 14: Encontra

# INPE 03 (seccion campaña)

In [ ]:
import pandas as pd
import numpy as np
import time
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def configurar_driver():
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument('--disable-browser-side-navigation')
    chrome_options.add_argument('--disable-gpu')

    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=chrome_options)

# URL DE CAMPAÑAS
base_url = "https://www.gob.pe/institucion/inpe/buscador?contenido=campa%C3%B1as&institucion=inpe&sort_by=recent&sheet="

all_campanas = []
path_folder = './Modulo_Tesis_Scrapping'
os.makedirs(path_folder, exist_ok=True)

# hay 367 campañas totales (aprox 15 páginas si son 25 por pág)
# Ajustamos el rango
for page in np.arange(1, 16):
    print(f"Extrayendo Campañas - Página {page}...")
    driver = configurar_driver()
    wait = WebDriverWait(driver, 30)

    try:
        driver.get(f"{base_url}{page}")

        # Esperamos a que los 'article' de las campañas carguen
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "article")))
        time.sleep(4)

        bloques = driver.find_elements(By.TAG_NAME, 'article')
        print(f"Página {page}: Encontradas {len(bloques)} campañas.")

        for bloque in bloques:
            try:
                # Título y Enlace
                tag_a = bloque.find_element(By.CLASS_NAME, "link-principal")

                # En campañas, el resumen suele estar en un contenedor de texto gris o el último div
                # Usamos el selector para asegurar compatibilidad
                resumen = bloque.find_element(By.XPATH, ".//div[last()]").text.strip()

                all_campanas.append({
                    "TIPO_CONTENIDO": "CAMPAÑA",
                    "PAGINA": page,
                    "TITULO_CAMPANA": tag_a.text.strip(),
                    "URL_CAMPANA": tag_a.get_attribute('href'),
                    "DESCRIPCION_CAMPANA": resumen
                })
            except:
                continue

    except Exception as e:
        print(f"Error en página {page}: {e}")
    finally:
        driver.quit()
        time.sleep(1)

# Guardado final de Campañas
if all_campanas:
    df_campanas = pd.DataFrame(all_campanas)
    filename = os.path.join(path_folder, 'Data_INPE_Campanas_Masiva.xlsx')
    df_campanas.to_excel(filename, index=False)

    print(f"\n¡PROCESO FINALIZADO!")
    print(f"Total de campañas capturadas: {len(df_campanas)}")
    print(f"Archivo guardado en: {filename}")
    print("\n--- PRIMERAS CAMPAÑAS ENCONTRADAS ---")
    print(df_campanas[['TITULO_CAMPANA', 'DESCRIPCION_CAMPANA']].head())

Extrayendo Campañas - Página 1...
Página 1: Encontradas 25 campañas.
Extrayendo Campañas - Página 2...
Página 2: Encontradas 25 campañas.
Extrayendo Campañas - Página 3...
Página 3: Encontradas 25 campañas.
Extrayendo Campañas - Página 4...
Página 4: Encontradas 25 campañas.
Extrayendo Campañas - Página 5...
Página 5: Encontradas 25 campañas.
Extrayendo Campañas - Página 6...
Página 6: Encontradas 25 campañas.
Extrayendo Campañas - Página 7...
Página 7: Encontradas 25 campañas.
Extrayendo Campañas - Página 8...
Página 8: Encontradas 25 campañas.
Extrayendo Campañas - Página 9...
Página 9: Encontradas 25 campañas.
Extrayendo Campañas - Página 10...
Página 10: Encontradas 25 campañas.
Extrayendo Campañas - Página 11...
Página 11: Encontradas 25 campañas.
Extrayendo Campañas - Página 12...
Página 12: Encontradas 25 campañas.
Extrayendo Campañas - Página 13...
Página 13: Encontradas 25 campañas.
Extrayendo Campañas - Página 14...
Página 14: Encontradas 25 campañas.
Extrayendo Campañas - Pá